<a href="https://colab.research.google.com/github/Ngga24/Learning-PreProcessing-Data-Mining/blob/main/2318120_DataCleansing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Import Library &  Mount Google Drive

In [19]:
import os
import numpy as np
import pandas as pd
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# 2. Load Dataset Mentah

In [20]:
# Path file mentah di Google Drive (sesuaikan nama folder)
raw_path = '/content/drive/MyDrive/Colab Notebooks/titanic_raw_train.csv'

# Load dataset
df = pd.read_csv(raw_path)

print("Dataset mentah berhasil dimuat!")
print("Ukuran data awal (baris, kolom):", df.shape)

Dataset mentah berhasil dimuat!
Ukuran data awal (baris, kolom): (891, 12)


# 3. Handling Missing Values

In [21]:
# Fillna Embarked dengan mode
df['Embarked'].fillna(df['Embarked'].mode()[0], inplace=True)

# Fillna Age dengan median berdasarkan Pclass
df['Age'] = df.groupby('Pclass')['Age'].transform(lambda x: x.fillna(x.median()))

# Transformasi Cabin menjadi boolean indicator
df['Has_Cabin'] = df['Cabin'].notnull().astype(int)

# Pastikan tidak ada missing value tersisa di kolom vital
print("Sisa missing value:")
print(df[['Embarked', 'Age', 'Has_Cabin']].isnull().sum())

Sisa missing value:
Embarked     0
Age          0
Has_Cabin    0
dtype: int64


/tmp/ipykernel_4411/2879109069.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Embarked'].fillna(df['Embarked'].mode()[0], inplace=True)


# 4. Feature Engineering

In [25]:
# Ekstraksi Gelar (Title)
df['Title'] = df['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
rare_titles = ['Lady', 'Countess', 'Capt', 'Col', 'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona']
df['Title'] = df['Title'].replace(rare_titles, 'Rare')
df['Title'] = df['Title'].replace(['Mlle', 'Ms'], 'Miss')
df['Title'] = df['Title'].replace('Mme', 'Mrs')

# Ukuran Keluarga & Indikator Bepergian Sendiri
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

# Log Transformation pada Fare
df['Fare_Log'] = np.log1p(df['Fare'])

# Age Grouping
bins = [0, 12, 18, 60, 100]
labels = ['Child', 'Teen', 'Adult', 'Senior']
df['Age_Group'] = pd.cut(df['Age'], bins=bins, labels=labels, right=False)

print("Feature Engineering berhasil dibuat!\n")

df[['Title', 'FamilySize', 'IsAlone', 'Fare_Log', 'Age_Group']].head()

Feature Engineering berhasil dibuat!



,Title,FamilySize,IsAlone,Fare_Log,Age_Group
0,Mr,2,0,2.110213,Adult
1,Mrs,2,0,4.280593,Adult
2,Miss,1,1,2.188856,Adult
3,Mrs,2,0,3.990834,Adult
4,Mr,1,1,2.202765,Adult


# 5. Categorical Encoding & Hapus Kolom Non-Prediktif

In [23]:
# Map Sex ke Numerik (0 = female, 1 = male)
df['Sex'] = df['Sex'].map({'female': 0, 'male': 1})

# One-Hot Encoding untuk Embarked, Title, dan Age_Group
df = pd.get_dummies(df, columns=['Embarked', 'Title', 'Age_Group'], drop_first=True, dtype=int)

# Hapus Kolom Mentah / Identifier / Redundat
columns_to_drop = ['PassengerId', 'Name', 'Ticket', 'Cabin', 'Fare']
df_clean = df.drop(columns=columns_to_drop, errors='ignore')

print("Encoding dan pembuangan kolom selesai!")

Encoding dan pembuangan kolom selesai!


# 6. Export Data Bersih ke Google Drive

In [24]:
# Tentukan folder dan nama file tujuan di Google Drive
folder_tujuan = '/content/drive/MyDrive/Colab Notebooks'
nama_file = 'titanic_train_clean.csv'

# Buat folder jika belum ada
os.makedirs(folder_tujuan, exist_ok=True)
full_path = os.path.join(folder_tujuan, nama_file)

# Export DataFrame bersih ke CSV
df_clean.to_csv(full_path, index=False)

# Tampilkan status dan 5 baris pertama
print("=== Hasil Akhit Cleansing Dataset ===")
print(f"File berhasil disimpan di: {full_path}")
print("Ukuran data bersih (baris, kolom):", df_clean.shape)

df_clean.head()

=== Hasil Akhit Cleansing Dataset ===
File berhasil disimpan di: /content/drive/MyDrive/Colab Notebooks/titanic_train_clean.csv
Ukuran data bersih (baris, kolom): (891, 19)


,Survived,Pclass,Sex,Age,SibSp,Parch,Has_Cabin,FamilySize,IsAlone,Fare_Log,Embarked_Q,Embarked_S,Title_Miss,Title_Mr,Title_Mrs,Title_Rare,Age_Group_Teen,Age_Group_Adult,Age_Group_Senior
0,0,3,1,22.0,1,0,0,2,0,2.110213,0,1,0,1,0,0,0,1,0
1,1,1,0,38.0,1,0,1,2,0,4.280593,0,0,0,0,1,0,0,1,0
2,1,3,0,26.0,0,0,0,1,1,2.188856,0,1,1,0,0,0,0,1,0
3,1,1,0,35.0,1,0,1,2,0,3.990834,0,1,0,0,1,0,0,1,0
4,0,3,1,35.0,0,0,0,1,1,2.202765,0,1,0,1,0,0,0,1,0
